<a href="https://colab.research.google.com/github/eze123-tech/Task-2-ezekiel/blob/main/DECODES_LABS_PROJECT_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [ ]:
data = pd.read_excel('sales_data_original.xlsx')
print(data)

# Clean column names by stripping whitespace to prevent KeyError due to subtle differences
data.columns = data.columns.str.strip()

# Drop rows where the target variable 'Sales' is NaN
data.dropna(subset=['Sales'], inplace=True)

x = data.drop('Sales', axis=1)
y = data['Sales']

              Segment    Country    Product  ... Month Number  Month Name    Year
0          Government     Canada  Carretera  ...          1.0         Jan  2019.0
1          Government    GERMANY  Carretera  ...          1.0         Jan  2019.0
2           Midmarket     France  Carretera  ...          6.0         Jun  2019.0
3           Midmarket    GERMANY  Carretera  ...          6.0         Jun  2019.0
4           Midmarket     Mexico  Carretera  ...          6.0         Jun  2019.0
..                ...        ...        ...  ...          ...         ...     ...
702    Small Business     France   Amarilla  ...          3.0         Mar  2019.0
703    Small Business     Mexico   Amarilla  ...         10.0         Oct  2019.0
704        Government     Mexico    Montana  ...          2.0         Feb  2019.0
705        Government     Canada      Paseo  ...          4.0         Apr  2019.0
706  Channel Partners        USA        VTT  ...          5.0         May  2019.0

[707 rows x 15 

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25, random_state = 42)

# Discretize the 'Sales' column into categories
# Using qcut to create bins with approximately equal number of observations
y_train = pd.qcut(y_train, q=3, labels=['Low', 'Medium', 'High'])
y_test = pd.qcut(y_test, q=3, labels=['Low', 'Medium', 'High'])

In [ ]:
# Identify categorical columns from x_train
categorical_cols = x_train.select_dtypes(include=['object']).columns

# Convert categorical columns to numerical using one-hot encoding
x_train_processed = pd.get_dummies(x_train, columns=categorical_cols, drop_first=True)
x_test_processed = pd.get_dummies(x_test, columns=categorical_cols, drop_first=True)

# Ensure x_test_processed has the same columns as x_train_processed, filling with 0 if a column is missing
# and dropping columns in x_test_processed that are not in x_train_processed
missing_cols_in_test = set(x_train_processed.columns) - set(x_test_processed.columns)
for c in missing_cols_in_test:
    x_test_processed[c] = 0
# Reorder columns to match x_train_processed
x_test_processed = x_test_processed[x_train_processed.columns]

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_processed)
x_test_scaled = scaler.transform(x_test_processed)

#base model on original data
clf_base = RandomForestClassifier(random_state= 42, n_jobs = -1)
clf_base.fit(x_train_scaled, y_train)
y_pred_base = clf_base.predict(x_test_scaled)
print('baseline classification report')
print(classification_report(y_test, y_pred_base))

baseline classification report
              precision    recall  f1-score   support

        High       1.00      0.83      0.91        59
         Low       1.00      0.90      0.95        59
      Medium       0.79      1.00      0.88        59

    accuracy                           0.91       177
   macro avg       0.93      0.91      0.91       177
weighted avg       0.93      0.91      0.91       177



In [ ]:
sm = SMOTE(random_state = 42)
x_resampled, y_resampled = sm.fit_resample(x_train_scaled, y_train)
print('after smote resampling')
print(pd.Series(y_resampled).value_counts())

after smote resampling
Sales
Low       176
Medium    176
High      176
Name: count, dtype: int64


In [ ]:
clf_smote = RandomForestClassifier(random_state = 42, n_jobs = -1)
clf_smote.fit(x_resampled, y_resampled)
y_pred_smote = clf_smote.predict(x_test_scaled)
print(classification_report(y_test, y_pred_smote))

              precision    recall  f1-score   support

        High       1.00      0.83      0.91        59
         Low       1.00      0.90      0.95        59
      Medium       0.79      1.00      0.88        59

    accuracy                           0.91       177
   macro avg       0.93      0.91      0.91       177
weighted avg       0.93      0.91      0.91       177

